# Chat with Miso 🐾

Load a pretrained MeowLLM checkpoint and talk to Miso.

**Runtime:** CPU is fine. GPU makes generation slightly faster, but the
model is tiny.

**Total time to first chat:** ~30 seconds.

## 1. Setup

In [ ]:
!git clone https://github.com/phanii9/MeowLLM.git
%cd MeowLLM
!pip install -q -e .

## 2. Get a checkpoint

You need a trained `best.pt` and `tokenizer.json`. Options:

**Option A — use a pretrained checkpoint from Hugging Face:**
```python
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(repo_id='hunt3rx99/meowllm', filename='best.pt')
tok = hf_hub_download(repo_id='hunt3rx99/meowllm', filename='tokenizer.json')
```

**Option B — use your own from the training notebook:**
Upload `best.pt` and `tokenizer.json` via the Colab file browser (left
sidebar), then set:
```python
ckpt = 'best.pt'
tok = 'tokenizer.json'
```

**Option C — train fresh right here:**

In [ ]:
# Try to download a pretrained checkpoint from Hugging Face first.
# If that fails (e.g. no pretrained release published yet), fall back to
# training from scratch right here in the notebook (~20 min on GPU T4).

HF_REPO = 'hunt3rx99/meowllm'  # change if you publish to a different repo

def _train_from_scratch():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'meow.generate_data',
                    '--out-dir', 'data', '--n', '20000', '--seed', '42'],
                   check=True)
    subprocess.run([sys.executable, '-m', 'meow.tokenizer', 'train',
                    'data/train.jsonl', 'data/tokenizer.json'], check=True)
    subprocess.run([sys.executable, '-m', 'meow.train', '--epochs', '10'],
                   check=True)
    return 'checkpoints/best.pt', 'data/tokenizer.json'

try:
    from huggingface_hub import hf_hub_download
    print(f'trying to download from hf.co/{HF_REPO}...')
    ckpt = hf_hub_download(repo_id=HF_REPO, filename='best.pt')
    tok = hf_hub_download(repo_id=HF_REPO, filename='tokenizer.json')
    print(f'downloaded ckpt: {ckpt}')
    print(f'downloaded tok:  {tok}')
except Exception as e:
    print(f'hf download failed: {e}')
    print('falling back to training from scratch (~20 min on GPU)')
    ckpt, tok = _train_from_scratch()

## 3. Load the model

In [ ]:
import torch
from meow.inference import load_model, chat_once
from meow.tokenizer import MeowTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, cfg = load_model(ckpt, device=device)
tokenizer = MeowTokenizer.from_file(tok)
print(f'loaded: {model.num_parameters()/1e6:.2f}M params on {device}')

## 4. Chat

In [ ]:
def say(prompt, temperature=0.8, top_k=40):
    response = chat_once(model, tokenizer, prompt,
                         device=device,
                         temperature=temperature, top_k=top_k)
    print(f'you:  {prompt}')
    print(f'miso: {response}\n')

# Try some prompts
say('hi miso')
say('are you hungry')
say('what do you see out the window')
say('the vacuum is coming out')
say('i love you')
say('what is the capital of france')

## 5. Interactive loop

In [ ]:
# Run this cell and type messages to Miso
while True:
    try:
        prompt = input('you:  ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not prompt or prompt.lower() in {'quit', 'exit', 'q'}:
        break
    say(prompt)

## About Miso

Miso is a ~3.5M parameter decoder-only transformer. The entire
personality is baked into the weights — there is no system prompt.

Miso knows about 15 things: food, naps, boxes, windows, birds, humans,
dogs, the vacuum, rain, affection, territory, and a handful of other
cat concerns. Ask Miso about anything else and it will stay in
character and dismiss you.

See [github.com/phanii9/MeowLLM](https://github.com/phanii9/MeowLLM)
for the full code.